# Day 2 — Feature Engineering & First XGBoost Model
## Objective
Build and evaluate our first XGBoost regression model
to predict delay_ratio.
## Baseline
Lane median with global median fallback.
Test MAE: 0.7405

In [14]:
import pandas as pd
import numpy as np
import xgboost as xgb
import sklearn
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("XGBoost version:", xgb.__version__)
print("Scikit-learn version:", sklearn.__version__)

Pandas version: 2.3.3
NumPy version: 2.2.6
XGBoost version: 3.2.0
Scikit-learn version: 1.7.2


In [15]:
from pathlib import Path
import pandas as pd

DATA = Path("../data/processed")

# Load the original dataset
legs = pd.read_csv(
    DATA / "legs.csv",
    parse_dates=["od_start_time", "trip_creation_time"]
)

# Parse od_end_time explicitly (formats can be mixed in the raw file)
legs["od_end_time"] = pd.to_datetime(legs["od_end_time"], format="mixed")
assert legs["od_end_time"].notna().all(), "Unparseable od_end_time values"

# Load the master split
split_ids = pd.read_csv(
    DATA / "split_trip_ids.csv"
)

# Validate trip assignments
assert split_ids["trip_uuid"].is_unique
assert set(split_ids["split"].unique()) == {"train", "calibration", "test"}

# Join master split to the leg dataset
df = legs.merge(
    split_ids[["trip_uuid", "split"]],
    on="trip_uuid",
    how="left",
    validate="many_to_one"
)

assert df["split"].notna().all(), "Some trips are missing split assignments"

# Create partitions
train = df[df["split"] == "train"].copy()
cal = df[df["split"] == "calibration"].copy()
test = df[df["split"] == "test"].copy()

for name, part in [
    ("Training", train),
    ("Calibration", cal),
    ("Test", test)
]:
    print(
        name,
        "| Legs:", len(part),
        "| Trips:", part["trip_uuid"].nunique()
    )

# Master split row-count and trip-count assertions
assert len(train) == 17674 and train["trip_uuid"].nunique() == 9969
assert len(cal) == 4301 and cal["trip_uuid"].nunique() == 2424
assert len(test) == 4394 and test["trip_uuid"].nunique() == 2424
assert len(train) + len(cal) + len(test) == len(df)
assert set(train["trip_uuid"]).isdisjoint(set(cal["trip_uuid"]))
assert set(train["trip_uuid"]).isdisjoint(set(test["trip_uuid"]))
assert set(cal["trip_uuid"]).isdisjoint(set(test["trip_uuid"]))

Training | Legs: 17674 | Trips: 9969
Calibration | Legs: 4301 | Trips: 2424
Test | Legs: 4394 | Trips: 2424


In [16]:
train["route_type"].value_counts(dropna=False)

route_type
FTL        9627
Carting    8047
Name: count, dtype: int64

In [17]:
def build_features(df):
    start = pd.to_datetime(df["od_start_time"])

    X = pd.DataFrame(index=df.index)

    X["osrm_time_min"] = df["osrm_time_min"]
    X["osrm_distance_km"] = df["osrm_distance_km"]
    X["osrm_speed_kmh"] = (
        df["osrm_distance_km"] / (df["osrm_time_min"] / 60)
    )
    X["log_osrm_time"] = np.log1p(df["osrm_time_min"])

    X["start_hour"] = start.dt.hour
    X["start_dow"] = start.dt.dayofweek
    X["is_weekend"] = (start.dt.dayofweek >= 5).astype(int)

    X["is_ftl"] = (df["route_type"] == "FTL").astype(int)

    return X

In [18]:
X_train = build_features(train)
X_cal = build_features(cal)
X_test = build_features(test)

print("Training features:", X_train.shape)
print("Calibration features:", X_cal.shape)
print("Test features:", X_test.shape)

print("\nMissing values in training features:")
print(X_train.isnull().sum())

print("\nFeature summary:")
display(X_train.describe())

Training features: (17674, 8)
Calibration features: (4301, 8)
Test features: (4394, 8)

Missing values in training features:
osrm_time_min       0
osrm_distance_km    0
osrm_speed_kmh      0
log_osrm_time       0
start_hour          0
start_dow           0
is_weekend          0
is_ftl              0
dtype: int64

Feature summary:


,osrm_time_min,osrm_distance_km,osrm_speed_kmh,log_osrm_time,start_hour,start_dow,is_weekend,is_ftl
count,17674.000000,17674.000000,17674.000000,17674.000000,17674.000000,17674.000000,17674.000000,17674.000000
mean,92.646656,117.042154,70.485174,3.912254,9.038927,2.974935,0.278432,0.544698
std,183.271250,250.465083,11.673241,0.935616,7.498261,1.948321,0.448240,0.498012
min,6.000000,9.072900,25.828056,1.945910,0.000000,0.000000,0.000000,0.000000
25%,25.000000,28.442950,62.433627,3.258097,3.000000,1.000000,0.000000,0.000000
50%,41.000000,44.697750,71.392007,3.737670,6.000000,3.000000,0.000000,1.000000
75%,76.000000,90.444300,81.019189,4.343805,16.000000,5.000000,1.000000,1.000000
max,1686.000000,2326.199100,96.907000,7.430707,23.000000,6.000000,1.000000,1.000000


In [19]:
from sklearn.metrics import mean_absolute_error, median_absolute_error
import numpy as np

def evaluate_model(y_true, y_pred, osrm_time):
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)

    excess_true = (y_true - 1) * osrm_time
    excess_pred = (y_pred - 1) * osrm_time

    excess_mae = mean_absolute_error(excess_true, excess_pred)

    return {
        "MAE_ratio": round(mae, 4),
        "MedAE_ratio": round(medae, 4),
        "MAE_excess_min": round(excess_mae, 2)
    }

In [20]:
y_train = train["delay_ratio"]
y_cal = cal["delay_ratio"]
y_test = test["delay_ratio"]

global_median = y_train.median()

lane_medians = train.groupby(
    ["source_center", "destination_center"]
)["delay_ratio"].median()

def predict_lane_median(df):
    lane_keys = pd.Series(
        list(zip(df["source_center"], df["destination_center"])),
        index=df.index
    )

    predictions = lane_keys.map(
        dict(zip(lane_medians.index, lane_medians.values))
    )

    return predictions.fillna(global_median).to_numpy()

cal_global_pred = np.full(len(cal), global_median)
test_global_pred = np.full(len(test), global_median)

cal_lane_pred = predict_lane_median(cal)
test_lane_pred = predict_lane_median(test)

baseline_results = {
    "Global Median - Calibration": evaluate_model(
        y_cal, cal_global_pred, cal["osrm_time_min"]
    ),
    "Lane Median - Calibration": evaluate_model(
        y_cal, cal_lane_pred, cal["osrm_time_min"]
    ),
    "Global Median - Test": evaluate_model(
        y_test, test_global_pred, test["osrm_time_min"]
    ),
    "Lane Median - Test": evaluate_model(
        y_test, test_lane_pred, test["osrm_time_min"]
    )
}

baseline_results

{'Global Median - Calibration': {'MAE_ratio': 1.0284,
  'MedAE_ratio': 0.45,
  'MAE_excess_min': 49.21},
 'Lane Median - Calibration': {'MAE_ratio': 0.6862,
  'MedAE_ratio': 0.25,
  'MAE_excess_min': 29.97},
 'Global Median - Test': {'MAE_ratio': 0.9847,
  'MedAE_ratio': 0.4444,
  'MAE_excess_min': 48.49},
 'Lane Median - Test': {'MAE_ratio': 0.7014,
  'MedAE_ratio': 0.2838,
  'MAE_excess_min': 32.21}}

In [21]:
train_trip_dates = (
    train.groupby("trip_uuid")["trip_creation_time"]
    .min()
    .sort_values()
)

split_index = int(len(train_trip_dates) * 0.8)

fit_trip_ids = train_trip_dates.index[:split_index]
val_trip_ids = train_trip_dates.index[split_index:]

fit_df = train[train["trip_uuid"].isin(fit_trip_ids)].copy()
val_df = train[train["trip_uuid"].isin(val_trip_ids)].copy()

assert set(fit_trip_ids).isdisjoint(set(val_trip_ids))
assert set(fit_trip_ids).union(set(val_trip_ids)) == set(train_trip_dates.index)

print("Fit trips:", fit_df["trip_uuid"].nunique())
print("Validation trips:", val_df["trip_uuid"].nunique())
print("Fit legs:", len(fit_df))
print("Validation legs:", len(val_df))
print("Fit creation end:", fit_df["trip_creation_time"].max())
print("Validation creation start:", val_df["trip_creation_time"].min())

Fit trips: 7975
Validation trips: 1994
Fit legs: 14073
Validation legs: 3601
Fit creation end: 2018-09-22 23:54:08.967657
Validation creation start: 2018-09-22 23:54:45.434611


In [22]:
X_fit = build_features(fit_df)
X_val = build_features(val_df)

y_fit = fit_df["delay_ratio"]
y_val = val_df["delay_ratio"]

print("X_fit:", X_fit.shape)
print("X_val:", X_val.shape)

print("Missing values in X_fit:", X_fit.isnull().sum().sum())
print("Missing values in X_val:", X_val.isnull().sum().sum())

X_fit: (14073, 8)
X_val: (3601, 8)
Missing values in X_fit: 0
Missing values in X_val: 0


In [23]:
from xgboost import XGBRegressor

xgb_v0 = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method="hist",
    random_state=42,
    eval_metric="mae",
    early_stopping_rounds=50
)

xgb_v0.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Best iteration:", xgb_v0.best_iteration)
print("Best validation MAE:", xgb_v0.best_score)

Best iteration: 1312
Best validation MAE: 0.8535363502434247


In [24]:
v0_val_pred = xgb_v0.predict(X_val)

v0_val_results = evaluate_model(
    y_val,
    v0_val_pred,
    val_df["osrm_time_min"]
)

v0_val_results

{'MAE_ratio': 0.8535, 'MedAE_ratio': 0.347, 'MAE_excess_min': 42.87}

In [25]:
val_lane_pred = predict_lane_median(val_df)

val_lane_results = evaluate_model(
    y_val,
    val_lane_pred,
    val_df["osrm_time_min"]
)

print("V0 XGBoost:", v0_val_results)
print("Lane Median:", val_lane_results)

V0 XGBoost: {'MAE_ratio': 0.8535, 'MedAE_ratio': 0.347, 'MAE_excess_min': 42.87}
Lane Median: {'MAE_ratio': 0.5275, 'MedAE_ratio': 0.1731, 'MAE_excess_min': 25.87}


In [26]:
import json
from pathlib import Path
from datetime import datetime, timezone

experiment_dir = Path("../experiments/day2/master_split")
experiment_dir.mkdir(parents=True, exist_ok=True)

v0_record = {
    "experiment": "V0",
    "description": "Initial XGBoost with 8 engineered features",
    "features": list(X_fit.columns),
    "fit_legs": len(fit_df),
    "validation_legs": len(val_df),
    "best_iteration": int(xgb_v0.best_iteration),
    "best_validation_mae": float(xgb_v0.best_score),
    "xgboost_validation_metrics": v0_val_results,
    "lane_median_validation_metrics": val_lane_results,
    "model_parameters": xgb_v0.get_params(),
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}
save_path = experiment_dir / "v0_results_v2.json"

if save_path.exists():
    raise FileExistsError(f"{save_path} already exists. Choose a new version.")

with open(save_path, "w") as f:
    json.dump(v0_record, f, indent=4, default=str)

print("Saved:", save_path)

Saved: ..\experiments\day2\master_split\v0_results_v2.json


In [27]:
print("Unique source hubs:", train["source_center"].nunique())
print("Unique destination hubs:", train["destination_center"].nunique())

print("\nMissing source hubs:", train["source_center"].isnull().sum())
print("Missing destination hubs:", train["destination_center"].isnull().sum())

print("\nSample source hubs:")
print(train["source_center"].unique()[:10])

print("\nSample destination hubs:")
print(train["destination_center"].unique()[:10])

Unique source hubs: 1415
Unique destination hubs: 1390

Missing source hubs: 0
Missing destination hubs: 0

Sample source hubs:
['IND209304AAA' 'IND462022AAA' 'IND561203AAB' 'IND572101AAA'
 'IND000000ACB' 'IND562132AAA' 'IND400072AAB' 'IND583101AAA'
 'IND583119AAA' 'IND583201AAA']

Sample destination hubs:
['IND000000ACB' 'IND209304AAA' 'IND562101AAA' 'IND561203AAB'
 'IND160002AAC' 'IND401104AAA' 'IND583201AAA' 'IND583101AAA'
 'IND583119AAA' 'IND602105AAB']


In [28]:
def build_features_v1(df):
    X = build_features(df).copy()

    X["source_center"] = df["source_center"].astype("category")
    X["destination_center"] = df["destination_center"].astype("category")

    return X

In [29]:
hub_categories = {
    "source_center": train["source_center"].unique(),
    "destination_center": train["destination_center"].unique()
}

def apply_hub_categories(X):
    X = X.copy()

    for col, categories in hub_categories.items():
        X[col] = pd.Categorical(
            X[col],
            categories=categories
        )

    return X

X_fit_v1 = apply_hub_categories(build_features_v1(fit_df))
X_val_v1 = apply_hub_categories(build_features_v1(val_df))
X_cal_v1 = apply_hub_categories(build_features_v1(cal))
X_test_v1 = apply_hub_categories(build_features_v1(test))

print("V1 feature shape:", X_fit_v1.shape)
print("V1 feature columns:", list(X_fit_v1.columns))
print("Unknown source hubs in validation:",
      X_val_v1["source_center"].isna().sum())
print("Unknown destination hubs in validation:",
      X_val_v1["destination_center"].isna().sum())

V1 feature shape: (14073, 10)
V1 feature columns: ['osrm_time_min', 'osrm_distance_km', 'osrm_speed_kmh', 'log_osrm_time', 'start_hour', 'start_dow', 'is_weekend', 'is_ftl', 'source_center', 'destination_center']
Unknown source hubs in validation: 0
Unknown destination hubs in validation: 0


In [30]:
xgb_v1 = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    eval_metric="mae",
    early_stopping_rounds=50
)

xgb_v1.fit(
    X_fit_v1,
    y_fit,
    eval_set=[(X_val_v1, y_val)],
    verbose=False
)

print("Best iteration:", xgb_v1.best_iteration)
print("Best validation MAE:", xgb_v1.best_score)

Best iteration: 1998
Best validation MAE: 0.7263825584686123


In [31]:
v1_val_pred = xgb_v1.predict(X_val_v1)

v1_val_results = evaluate_model(
    y_val,
    v1_val_pred,
    val_df["osrm_time_min"]
)

print("V0 XGBoost:", v0_val_results)
print("V1 XGBoost:", v1_val_results)
print("Lane Median:", val_lane_results)

V0 XGBoost: {'MAE_ratio': 0.8535, 'MedAE_ratio': 0.347, 'MAE_excess_min': 42.87}
V1 XGBoost: {'MAE_ratio': 0.9548, 'MedAE_ratio': 0.2587, 'MAE_excess_min': 49.25}
Lane Median: {'MAE_ratio': 0.5275, 'MedAE_ratio': 0.1731, 'MAE_excess_min': 25.87}


In [32]:
v1_record = {
    "experiment": "V1",
    "description": "XGBoost with 8 engineered features and 2 hub categorical features",
    "features": list(X_fit_v1.columns),
    "fit_legs": len(fit_df),
    "validation_legs": len(val_df),
    "best_iteration": int(xgb_v1.best_iteration),
    "best_validation_mae": float(xgb_v1.best_score),
    "xgboost_validation_metrics": v1_val_results,
    "v0_validation_metrics": v0_val_results,
    "lane_median_validation_metrics": val_lane_results,
    "model_parameters": xgb_v1.get_params()
}

save_path = experiment_dir / "v1_results_v1.json"

if save_path.exists():
    raise FileExistsError(f"{save_path} already exists. Choose another version.")

with open(save_path, "w") as f:
    json.dump(v1_record, f, indent=4, default=str)

print("Saved:", save_path)

Saved: ..\experiments\day2\master_split\v1_results_v1.json


In [33]:
lane_history = (
    fit_df.groupby(["source_center", "destination_center"])["delay_ratio"]
    .agg(["median", "count"])
    .reset_index()
)

lane_history.columns = [
    "source_center",
    "destination_center",
    "lane_median_ratio",
    "lane_history_count"
]

print("Historical lanes:", len(lane_history))
print("\nSample:")
display(lane_history.head())

Historical lanes: 2352

Sample:


,source_center,destination_center,lane_median_ratio,lane_history_count
0,IND000000AAL,IND411033AAA,2.570833,14
1,IND000000AAS,IND783370AAC,2.566667,5
2,IND000000AAZ,IND444203AAA,6.020833,1
3,IND000000ABA,IND683565AAA,1.000000,3
4,IND000000ABD,IND562132AAA,1.708108,9


In [34]:
def add_historical_lane_features(df, history_data=None, is_fit=False):
    result = df.copy()
    lane_cols = ["source_center", "destination_center"]

    if is_fit:
        result["lane_median_ratio"] = 1.0
        result["lane_history_count"] = 0

        for _, group in result.groupby(lane_cols, sort=False):
            group = group.sort_values("od_start_time")

            completed = group.sort_values("od_end_time")

            completed_delays = completed["delay_ratio"].to_numpy()

            completed_ends = pd.to_datetime(
              completed["od_end_time"],
              format="mixed"
             ).to_numpy(dtype="datetime64[ns]")

            pointer = 0
            past_delays = []

            for idx, row in group.iterrows():

                current_start = np.datetime64(
                    pd.Timestamp(row["od_start_time"]), "ns"
                )

                while (
                    pointer < len(completed)
                    and completed_ends[pointer] <= current_start
                ):
                    past_delays.append(completed_delays[pointer])
                    pointer += 1

                result.loc[idx, "lane_history_count"] = len(past_delays)

                if past_delays:
                    result.loc[idx, "lane_median_ratio"] = np.median(
                        past_delays
                    )

        return result

    if history_data is None:
        raise ValueError("Historical data is required.")

    lookup = history_data.set_index(lane_cols)
    keys = pd.MultiIndex.from_frame(result[lane_cols])

    result["lane_median_ratio"] = (
        lookup["lane_median_ratio"]
        .reindex(keys)
        .fillna(1.0)
        .to_numpy()
    )

    result["lane_history_count"] = (
        lookup["lane_history_count"]
        .reindex(keys)
        .fillna(0)
        .to_numpy()
    )

    return result

In [35]:
fit_v2 = add_historical_lane_features(
    fit_df,
    is_fit=True
)

val_v2 = add_historical_lane_features(
    val_df,
    history_data=lane_history
)

cal_v2 = add_historical_lane_features(
    cal,
    history_data=lane_history
)

test_v2 = add_historical_lane_features(
    test,
    history_data=lane_history
)

In [36]:
datasets = {
    "Fit": fit_v2,
    "Validation": val_v2,
    "Calibration": cal_v2,
    "Test": test_v2
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Missing lane medians:", df["lane_median_ratio"].isna().sum())
    print("Missing history counts:", df["lane_history_count"].isna().sum())
    print("Minimum history count:", df["lane_history_count"].min())
    print("Maximum history count:", df["lane_history_count"].max())
    print("Neutral fallback lanes:", (df["lane_history_count"] == 0).sum())


Fit
Shape: (14073, 17)
Missing lane medians: 0
Missing history counts: 0
Minimum history count: 0
Maximum history count: 71
Neutral fallback lanes: 2437

Validation
Shape: (3601, 17)
Missing lane medians: 0
Missing history counts: 0
Minimum history count: 0.0
Maximum history count: 74.0
Neutral fallback lanes: 149

Calibration
Shape: (4301, 17)
Missing lane medians: 0
Missing history counts: 0
Minimum history count: 0.0
Maximum history count: 74.0
Neutral fallback lanes: 260

Test
Shape: (4394, 17)
Missing lane medians: 0
Missing history counts: 0
Minimum history count: 0.0
Maximum history count: 74.0
Neutral fallback lanes: 422


In [37]:
print("Fit departure range:")
print(fit_df["od_start_time"].min(), "to", fit_df["od_start_time"].max())

print("\nValidation departure range:")
print(val_df["od_start_time"].min(), "to", val_df["od_start_time"].max())

print("\nFit completion range:")
print(fit_df["od_end_time"].min(), "to", fit_df["od_end_time"].max())

print("\nValidation departure before latest fit completion:")
print(
    (val_df["od_start_time"] <= fit_df["od_end_time"].max()).sum()
)

Fit departure range:
2018-09-12 00:00:16.535741 to 2018-09-25 02:41:11.112662

Validation departure range:
2018-09-22 23:54:45.434611 to 2018-09-27 23:49:36.190915

Fit completion range:
2018-09-12 00:50:10.814399 to 2018-09-25 16:21:16.077674

Validation departure before latest fit completion:
2773


In [38]:
for name, df in {
    "Fit": fit_df,
    "Validation": val_df,
    "Calibration": cal,
    "Test": test
}.items():
    print(
        name,
        df["od_start_time"].dtype,
        df["od_end_time"].dtype
    )

Fit datetime64[ns] datetime64[ns]
Validation datetime64[ns] datetime64[ns]
Calibration datetime64[ns] datetime64[ns]
Test datetime64[ns] datetime64[ns]


In [39]:
def add_historical_lane_features_asof(target_df, history_df):
    result = target_df.copy()

    lane_cols = ["source_center", "destination_center"]

    result["lane_median_ratio"] = 1.0
    result["lane_history_count"] = 0

    history_groups = {
        key: group.sort_values("od_end_time")
        for key, group in history_df.groupby(lane_cols, sort=False)
    }

    for key, target_group in result.groupby(lane_cols, sort=False):

        if key not in history_groups:
            continue

        hist = history_groups[key]

        end_times = hist["od_end_time"].to_numpy(
            dtype="datetime64[ns]"
        )
        delays = hist["delay_ratio"].to_numpy()

        for idx, row in target_group.iterrows():

            start_time = np.datetime64(
                row["od_start_time"], "ns"
            )

            count = np.searchsorted(
                end_times, start_time, side="right"
            )

            result.loc[idx, "lane_history_count"] = count

            if count > 0:
                result.loc[idx, "lane_median_ratio"] = np.median(
                    delays[:count]
                )

    return result

In [40]:
fit_v2 = add_historical_lane_features_asof(fit_df, fit_df)

val_v2 = add_historical_lane_features_asof(val_df, fit_df)

cal_v2 = add_historical_lane_features_asof(cal, train)

test_history = pd.concat([train, cal], ignore_index=True)
test_v2 = add_historical_lane_features_asof(test, test_history)

print("Fit:", fit_v2.shape)
print("Validation:", val_v2.shape)
print("Calibration:", cal_v2.shape)
print("Test:", test_v2.shape)

Fit: (14073, 17)
Validation: (3601, 17)
Calibration: (4301, 17)
Test: (4394, 17)


In [41]:
for name, df in {
    "Fit": fit_v2,
    "Validation": val_v2,
    "Calibration": cal_v2,
    "Test": test_v2
}.items():
    print(
        name,
        "| Shape:", df.shape,
        "| Missing:", df[["lane_median_ratio", "lane_history_count"]].isna().sum().sum(),
        "| Fallback rows:", (df["lane_history_count"] == 0).sum(),
        "| Max history:", df["lane_history_count"].max()
    )

Fit | Shape: (14073, 17) | Missing: 0 | Fallback rows: 2437 | Max history: 71
Validation | Shape: (3601, 17) | Missing: 0 | Fallback rows: 149 | Max history: 74
Calibration | Shape: (4301, 17) | Missing: 0 | Fallback rows: 170 | Max history: 94
Test | Shape: (4394, 17) | Missing: 0 | Fallback rows: 240 | Max history: 121


In [42]:
def build_features_v2(df):
    X = build_features_v1(df)

    X["lane_median_ratio"] = df["lane_median_ratio"]
    X["lane_history_count"] = df["lane_history_count"]

    return X

In [43]:
X_fit_v2 = apply_hub_categories(build_features_v2(fit_v2))
X_val_v2 = apply_hub_categories(build_features_v2(val_v2))
X_cal_v2 = apply_hub_categories(build_features_v2(cal_v2))
X_test_v2 = apply_hub_categories(build_features_v2(test_v2))

print("Fit:", X_fit_v2.shape)
print("Validation:", X_val_v2.shape)
print("Calibration:", X_cal_v2.shape)
print("Test:", X_test_v2.shape)
print("\nFeatures:")
print(X_fit_v2.columns.tolist())

Fit: (14073, 12)
Validation: (3601, 12)
Calibration: (4301, 12)
Test: (4394, 12)

Features:
['osrm_time_min', 'osrm_distance_km', 'osrm_speed_kmh', 'log_osrm_time', 'start_hour', 'start_dow', 'is_weekend', 'is_ftl', 'source_center', 'destination_center', 'lane_median_ratio', 'lane_history_count']


In [44]:
model_v2 = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    eval_metric="mae",
    early_stopping_rounds=50
)

model_v2.fit(
    X_fit_v2,
    y_fit,
    eval_set=[(X_val_v2, y_val)],
    verbose=False
)

print("Best iteration:", model_v2.best_iteration)
print("Best validation MAE:", model_v2.best_score)

Best iteration: 1965
Best validation MAE: 0.7115336129133054


In [45]:
pred_v2 = model_v2.predict(X_val_v2)

v2_metrics = evaluate_model(
    y_val,
    pred_v2,
    val_df["osrm_time_min"]
)

print("V2 Validation Metrics:")
print(v2_metrics)

V2 Validation Metrics:
{'MAE_ratio': 0.8976, 'MedAE_ratio': 0.2496, 'MAE_excess_min': 45.78}


In [46]:
print("Fit lane-history distribution:")
print(fit_v2["lane_history_count"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
))

print("\nValidation lane-history distribution:")
print(val_v2["lane_history_count"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
))

Fit lane-history distribution:
count    14073.000000
mean         5.734740
std          8.457082
min          0.000000
25%          1.000000
50%          3.000000
75%          7.000000
90%         13.000000
95%         21.000000
max         71.000000
Name: lane_history_count, dtype: float64

Validation lane-history distribution:
count    3601.000000
mean       11.831713
std        13.527441
min         0.000000
25%         5.000000
50%         8.000000
75%        11.000000
90%        25.000000
95%        45.000000
max        74.000000
Name: lane_history_count, dtype: float64


In [47]:
lane_history_metrics = evaluate_model(
    y_val,
    val_v2["lane_median_ratio"],
    val_df["osrm_time_min"]
)

print("Historical lane feature alone:")
print(lane_history_metrics)

print("\nLane history count correlation:")
print(
    val_v2[["lane_history_count", "delay_ratio"]]
    .corr(method="spearman")
)

Historical lane feature alone:
{'MAE_ratio': 0.6982, 'MedAE_ratio': 0.2524, 'MAE_excess_min': 35.75}

Lane history count correlation:
                    lane_history_count  delay_ratio
lane_history_count            1.000000    -0.023381
delay_ratio                  -0.023381     1.000000


In [48]:
print(
    lane_history["lane_history_count"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

count    2352.000000
mean        5.983418
std         6.343706
min         1.000000
25%         2.000000
50%         5.000000
75%         8.000000
90%        11.000000
95%        15.000000
max        74.000000
Name: lane_history_count, dtype: float64


In [49]:
lane_lookup = lane_history.set_index(
    ["source_center", "destination_center"]
)["lane_median_ratio"]

val_keys = pd.MultiIndex.from_frame(
    val_df[["source_center", "destination_center"]]
)

lane_median_val = (
    lane_lookup.reindex(val_keys)
    .fillna(global_median)
    .to_numpy()
)

print("Validation predictions:", len(lane_median_val))
print("Missing predictions:", pd.isna(lane_median_val).sum())

Validation predictions: 3601
Missing predictions: 0


In [50]:
comparison = pd.DataFrame({
    "original_lane_median": lane_median_val,
    "historical_lane_median": val_v2["lane_median_ratio"],
    "actual_delay_ratio": y_val
})

comparison["prediction_difference"] = (
    comparison["original_lane_median"]
    - comparison["historical_lane_median"]
).abs()

print(comparison.describe())
print(
    "\nPredictions with different values:",
    (comparison["prediction_difference"] > 0.000001).sum()
)

       original_lane_median  historical_lane_median  actual_delay_ratio  \
count           3601.000000             3601.000000         3601.000000   
mean               2.315671                2.274186            2.545389   
std                1.626429                1.646560            2.658839   
min                0.462114                0.462114            0.407895   
25%                1.709302                1.646104            1.656250   
50%                2.000000                1.968954            2.000000   
75%                2.428571                2.428571            2.575758   
max               28.638889               28.638889           66.388889   

       prediction_difference  
count            3601.000000  
mean                0.041623  
std                 0.199168  
min                 0.000000  
25%                 0.000000  
50%                 0.000000  
75%                 0.000000  
max                 1.000000  

Predictions with different values: 207


In [51]:
different = comparison[
    comparison["prediction_difference"] > 0.000001
].copy()

different["original_error"] = (
    different["original_lane_median"]
    - different["actual_delay_ratio"]
).abs()

different["historical_error"] = (
    different["historical_lane_median"]
    - different["actual_delay_ratio"]
).abs()

print(different[[
    "original_lane_median",
    "historical_lane_median",
    "actual_delay_ratio",
    "original_error",
    "historical_error"
]].head(10))

print("\nAverage original error:", different["original_error"].mean())
print("Average historical error:", different["historical_error"].mean())

       original_lane_median  historical_lane_median  actual_delay_ratio  \
14076              2.062062                2.026016            1.973896   
14077              2.000000                1.000000            1.459459   
14082              2.178019                2.116830            2.092879   
14086              2.272000                2.316000            2.204000   
14088              2.000000                1.000000            1.735294   
14106              2.000000                1.000000            2.357143   
14110              2.000000                1.000000            1.245902   
14144              1.868829                1.842031            1.881690   
14164              2.013158                1.894737            2.675000   
14169              2.000000                1.000000            1.527778   

       original_error  historical_error  
14076        0.088166          0.052120  
14077        0.540541          0.459459  
14082        0.085139          0.023951  
14086 

In [52]:
different_counts = val_v2.loc[
    different.index, "lane_history_count"
]

print("Differing predictions:", len(different))
print("Zero history:", (different_counts == 0).sum())
print("Some history:", (different_counts > 0).sum())

Differing predictions: 207
Zero history: 149
Some history: 58


In [53]:
zero_history_idx = val_v2.index[
    val_v2["lane_history_count"] == 0
]

original_fallback = lane_median_val[
    val_v2.index.get_indexer(zero_history_idx)
]

print("Zero-history validation legs:", len(zero_history_idx))
print(
    "Original lane median differs from 1.0:",
    (original_fallback != 1.0).sum()
)

Zero-history validation legs: 149
Original lane median differs from 1.0: 149


In [54]:
fit_v3 = fit_v2.copy()
val_v3 = val_v2.copy()

fit_v3["lane_median_ratio"] = np.where(
    fit_v3["lane_history_count"] == 0,
    fit_v3["lane_median_ratio"],
    fit_v3["lane_median_ratio"]
)

val_v3["lane_median_ratio"] = np.where(
    val_v3["lane_history_count"] == 0,
    lane_median_val,
    val_v3["lane_median_ratio"]
)

In [55]:
print("Zero-history rows before:", 
      (val_v2["lane_history_count"] == 0).sum())

print("Zero-history rows after:", 
      (val_v3["lane_history_count"] == 0).sum())

print("Fallback values updated:", 
      (val_v3["lane_median_ratio"] != val_v2["lane_median_ratio"]).sum())

Zero-history rows before: 149
Zero-history rows after: 149
Fallback values updated: 149


In [56]:
print("Unique source hubs:", fit_df["source_center"].nunique())
print("Unique source hubs in validation:", val_df["source_center"].nunique())

Unique source hubs: 1370
Unique source hubs in validation: 1057


In [57]:
source_history = (
    fit_df.groupby("source_center")["delay_ratio"]
    .agg(["median", "count"])
    .reset_index()
)

source_history.columns = [
    "source_center",
    "source_median_ratio",
    "source_history_count"
]

print("Source hubs:", len(source_history))
print(source_history.head())

Source hubs: 1370
  source_center  source_median_ratio  source_history_count
0  IND000000AAL             2.570833                    14
1  IND000000AAS             2.566667                     5
2  IND000000AAZ             6.020833                     1
3  IND000000ABA             1.000000                     3
4  IND000000ABD             1.708108                     9


In [58]:
source_history_asof = add_historical_lane_features_asof  # Keep existing lane function unchanged

print("Fit unique source hubs:", fit_df["source_center"].nunique())
print("Validation unique source hubs:", val_df["source_center"].nunique())

print(
    "Validation rows with zero lane history:",
    (val_v2["lane_history_count"] == 0).sum()
)

Fit unique source hubs: 1370
Validation unique source hubs: 1057
Validation rows with zero lane history: 149


In [59]:
source_groups = {
    key: group.sort_values("od_end_time")
    for key, group in fit_df.groupby("source_center", sort=False)
}

source_history_counts = []

for _, row in val_df.iterrows():
    hub = row["source_center"]

    if hub not in source_groups:
        source_history_counts.append(0)
        continue

    hist = source_groups[hub]
    end_times = hist["od_end_time"].to_numpy(dtype="datetime64[ns]")

    count = np.searchsorted(
        end_times,
        np.datetime64(row["od_start_time"], "ns"),
        side="right"
    )

    source_history_counts.append(count)

source_history_counts = np.array(source_history_counts)

print("Validation rows:", len(val_df))
print("Rows with source history:", (source_history_counts > 0).sum())
print("Rows without source history:", (source_history_counts == 0).sum())
print("Source history coverage:", round((source_history_counts > 0).mean() * 100, 2), "%")

Validation rows: 3601
Rows with source history: 3542
Rows without source history: 59
Source history coverage: 98.36 %


In [60]:
def add_source_history_asof(target_df, history_df):
    result = target_df.copy()
    result["source_median_ratio"] = 1.0
    result["source_history_count"] = 0

    history_groups = {
        key: group.sort_values("od_end_time")
        for key, group in history_df.groupby("source_center", sort=False)
    }

    for hub, target_group in result.groupby("source_center", sort=False):
        if hub not in history_groups:
            continue

        hist = history_groups[hub]

        end_times = hist["od_end_time"].to_numpy(
            dtype="datetime64[ns]"
        )
        delays = hist["delay_ratio"].to_numpy()

        for idx, row in target_group.iterrows():
            start_time = np.datetime64(row["od_start_time"], "ns")

            count = np.searchsorted(
                end_times,
                start_time,
                side="right"
            )

            result.loc[idx, "source_history_count"] = count

            if count > 0:
                result.loc[idx, "source_median_ratio"] = np.median(
                    delays[:count]
                )

    return result

In [61]:
fit_v3 = add_source_history_asof(fit_df, fit_df)

val_v3 = add_source_history_asof(val_df, fit_df)

cal_v3 = add_source_history_asof(cal, train)

test_v3 = add_source_history_asof(test, train)

In [62]:
for name, df in {
    "Fit": fit_v3,
    "Validation": val_v3,
    "Calibration": cal_v3,
    "Test": test_v3
}.items():

    zero_history = (df["source_history_count"] == 0).sum()

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Zero source-history rows:", zero_history)
    print("Rows with source history:", len(df) - zero_history)
    print("Median source history:", df["source_history_count"].median())


Fit
Rows: 14073
Zero source-history rows: 1545
Rows with source history: 12528
Median source history: 6.0

Validation
Rows: 3601
Zero source-history rows: 59
Rows with source history: 3542
Median source history: 13.0

Calibration
Rows: 4301
Zero source-history rows: 51
Rows with source history: 4250
Median source history: 16.0

Test
Rows: 4394
Zero source-history rows: 128
Rows with source history: 4266
Median source history: 14.0


In [63]:
def combine_history_features(lane_df, source_df):
    result = lane_df.copy()

    use_source = result["lane_history_count"] == 0

    result["combined_history_ratio"] = np.where(
        use_source,
        source_df["source_median_ratio"],
        result["lane_median_ratio"]
    )

    result["combined_history_count"] = np.where(
        use_source,
        source_df["source_history_count"],
        result["lane_history_count"]
    )

    result["history_level"] = np.select(
        [
            result["lane_history_count"] > 0,
            source_df["source_history_count"] > 0
        ],
        ["lane", "source"],
        default="global"
    )

    return result


fit_v3 = combine_history_features(fit_v2, fit_v3)
val_v3 = combine_history_features(val_v2, val_v3)
cal_v3 = combine_history_features(cal_v2, cal_v3)
test_v3 = combine_history_features(test_v2, test_v3)

In [64]:
for name, df in {
    "Fit": fit_v3,
    "Validation": val_v3,
    "Calibration": cal_v3,
    "Test": test_v3
}.items():

    print(f"\n{name}")
    print(df["history_level"].value_counts())


Fit
history_level
lane      11636
global     1545
source      892
Name: count, dtype: int64

Validation
history_level
lane      3452
source      90
global      59
Name: count, dtype: int64

Calibration
history_level
lane      4131
source     119
global      51
Name: count, dtype: int64

Test
history_level
lane      4154
source     145
global      95
Name: count, dtype: int64


In [65]:
def build_features_v3(df):
    X = build_features_v1(df)

    X["combined_history_ratio"] = df["combined_history_ratio"]
    X["combined_history_count"] = df["combined_history_count"]

    return X


X_fit_v3 = build_features_v3(fit_v3)
X_val_v3 = build_features_v3(val_v3)
X_cal_v3 = build_features_v3(cal_v3)
X_test_v3 = build_features_v3(test_v3)

print("Fit:", X_fit_v3.shape)
print("Validation:", X_val_v3.shape)
print("Calibration:", X_cal_v3.shape)
print("Test:", X_test_v3.shape)

print("\nMissing values:")
print(X_fit_v3.isna().sum().sum(),
      X_val_v3.isna().sum().sum(),
      X_cal_v3.isna().sum().sum(),
      X_test_v3.isna().sum().sum())

Fit: (14073, 12)
Validation: (3601, 12)
Calibration: (4301, 12)
Test: (4394, 12)

Missing values:
0 0 0 0


In [66]:
from xgboost import XGBRegressor

xgb_v3 = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    eval_metric="mae",
    early_stopping_rounds=50
)

xgb_v3.fit(
    X_fit_v3,
    y_fit,
    eval_set=[(X_val_v3, y_val)],
    verbose=False
)

print("Best iteration:", xgb_v3.best_iteration)
print("Best validation MAE:", xgb_v3.best_score)

Best iteration: 1019
Best validation MAE: 0.718879770327197


In [67]:
pred_v3 = xgb_v3.predict(X_val_v3)

metrics_v3 = evaluate_model(
    y_val,
    pred_v3,
    val_df["osrm_time_min"]
)

print(metrics_v3)

{'MAE_ratio': 0.7189, 'MedAE_ratio': 0.2525, 'MAE_excess_min': 35.18}


In [68]:
import sklearn
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

print("Scikit-learn:", sklearn.__version__)
print("Random Forest: Ready")
print("Extra Trees: Ready")

Scikit-learn: 1.7.2
Random Forest: Ready
Extra Trees: Ready


In [69]:
from sklearn.preprocessing import OrdinalEncoder

categorical_cols = ["source_center", "destination_center"]

X_fit_v4 = X_fit_v2.copy()
X_val_v4 = X_val_v2.copy()

encoder_v4 = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_fit_v4[categorical_cols] = encoder_v4.fit_transform(
    X_fit_v4[categorical_cols].astype(str)
)

X_val_v4[categorical_cols] = encoder_v4.transform(
    X_val_v4[categorical_cols].astype(str)
)

print("Fit:", X_fit_v4.shape)
print("Validation:", X_val_v4.shape)

Fit: (14073, 12)
Validation: (3601, 12)


In [70]:
from sklearn.ensemble import RandomForestRegressor

rf_v4 = RandomForestRegressor(
    n_estimators=400,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.8,
    n_jobs=-1,
    random_state=42
)

rf_v4.fit(X_fit_v4, y_fit)

print("Random Forest training completed!")

Random Forest training completed!


In [71]:
pred_v4 = rf_v4.predict(X_val_v4)

metrics_v4 = evaluate_model(
    y_val,
    pred_v4,
    val_df["osrm_time_min"]
)

print("V4 Random Forest:", metrics_v4)

V4 Random Forest: {'MAE_ratio': 0.7553, 'MedAE_ratio': 0.3342, 'MAE_excess_min': 37.72}


In [72]:
model_comparison = pd.DataFrame([
    {"Model": "V0 - Basic XGBoost", "MAE": 0.8981},
    {"Model": "V1 - Hub XGBoost", "MAE": 1.0344},
    {"Model": "V2 - Historical Lane XGBoost", "MAE": 0.7679},
    {"Model": "V3 - Lane + Source XGBoost", "MAE": 0.7746},
    {"Model": "V4 - Random Forest", "MAE": metrics_v4["MAE_ratio"]},
    {"Model": "Lane Median Baseline", "MAE": 0.5662}
])

model_comparison = model_comparison.sort_values("MAE")

display(model_comparison)

,Model,MAE
5,Lane Median Baseline,0.5662
4,V4 - Random Forest,0.7553
2,V2 - Historical Lane XGBoost,0.7679
3,V3 - Lane + Source XGBoost,0.7746
0,V0 - Basic XGBoost,0.8981
1,V1 - Hub XGBoost,1.0344


In [73]:
import json
from pathlib import Path

output_dir = Path("../experiments/day2/master_split")
output_dir.mkdir(parents=True, exist_ok=True)

result_v4 = {
    "model": "Random Forest Regressor",
    "features": "V2 historical lane features",
    "parameters": rf_v4.get_params(),
    "validation_metrics": metrics_v4
}

output_path = output_dir / "v4_random_forest_results_v1.json"

if output_path.exists():
    raise FileExistsError(f"{output_path} already exists. Choose a new version.")

with open(output_path, "w") as f:
    json.dump(result_v4, f, indent=4, default=str)

print("Saved:", output_path)

Saved: ..\experiments\day2\master_split\v4_random_forest_results_v1.json


In [74]:
from sklearn.ensemble import ExtraTreesRegressor

et_v5 = ExtraTreesRegressor(
    n_estimators=400,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.8,
    n_jobs=-1,
    random_state=42
)

et_v5.fit(X_fit_v4, y_fit)

print("Extra Trees training completed!")

Extra Trees training completed!


In [75]:
pred_v5 = et_v5.predict(X_val_v4)

metrics_v5 = evaluate_model(
    y_val,
    pred_v5,
    val_df["osrm_time_min"]
)

print("V5 Extra Trees:", metrics_v5)

V5 Extra Trees: {'MAE_ratio': 0.7259, 'MedAE_ratio': 0.3264, 'MAE_excess_min': 35.53}


In [76]:
result_v5 = {
    "model": "Extra Trees Regressor",
    "features": "V2 historical lane features",
    "parameters": et_v5.get_params(),
    "validation_metrics": metrics_v5
}

output_path = output_dir / "v5_extra_trees_results_v1.json"

if output_path.exists():
    raise FileExistsError(f"{output_path} already exists.")

with open(output_path, "w") as f:
    json.dump(result_v5, f, indent=4, default=str)

print("Saved:", output_path)

Saved: ..\experiments\day2\master_split\v5_extra_trees_results_v1.json


In [77]:
fair_lane_val = evaluate_model(y_val, lane_median_val, val_df["osrm_time_min"])
print("Fair lane baseline on validation:", fair_lane_val)

Fair lane baseline on validation: {'MAE_ratio': 0.6741, 'MedAE_ratio': 0.25, 'MAE_excess_min': 33.18}


In [78]:
rows = {
    "Lane median (fit-only, fair)": lane_median_val,
    "V0 basic":          xgb_v0.predict(X_val),
    "V1 hubs":           xgb_v1.predict(X_val_v1),
    "V2 lane history":   model_v2.predict(X_val_v2),
    "V3 lane + source":  xgb_v3.predict(X_val_v3),
    "V4 random forest":  rf_v4.predict(X_val_v4),
}
table = pd.DataFrame({
    k: evaluate_model(y_val, p, val_df["osrm_time_min"]) for k, p in rows.items()
}).T
table.sort_values("MAE_ratio")

,MAE_ratio,MedAE_ratio,MAE_excess_min
"Lane median (fit-only, fair)",0.6741,0.2500,33.18
V3 lane + source,0.7189,0.2525,35.18
V4 random forest,0.7553,0.3342,37.72
V0 basic,0.8535,0.3470,42.87
V2 lane history,0.8976,0.2496,45.78
V1 hubs,0.9548,0.2587,49.25


In [79]:
from sklearn.metrics import mean_absolute_error
for name, m, X in [("V1", xgb_v1, X_val_v1), ("V2", model_v2, X_val_v2)]:
    p = m.predict(X)
    print(name, "recomputed:", round(mean_absolute_error(y_val, p), 4),
          "| XGBoost best_score:", round(m.best_score, 4))

V1 recomputed: 0.9548 | XGBoost best_score: 0.7264
V2 recomputed: 0.8976 | XGBoost best_score: 0.7115


In [80]:
g_fit = float(fit_df["delay_ratio"].median())

num_cols = [
    "osrm_time_min",
    "osrm_distance_km",
    "osrm_speed_kmh",
    "log_osrm_time",
    "start_hour",
    "start_dow",
    "is_weekend",
    "is_ftl"
]

def base_pred(d):
    return (
        d["lane_median_ratio"]
        .where(d["lane_history_count"] > 0, g_fit)
        .to_numpy()
    )

def make_X(d):
    X = build_features(d)[num_cols].copy()
    X["lane_base"] = base_pred(d)
    X["lane_history_count"] = d["lane_history_count"].to_numpy()
    return X

base_fit = base_pred(fit_v2)
base_val = base_pred(val_v2)

Xf = make_X(fit_v2)
Xv = make_X(val_v2)

print("Features:", Xf.columns.tolist())
print("Fit:", Xf.shape, "| Validation:", Xv.shape)

Features: ['osrm_time_min', 'osrm_distance_km', 'osrm_speed_kmh', 'log_osrm_time', 'start_hour', 'start_dow', 'is_weekend', 'is_ftl', 'lane_base', 'lane_history_count']
Fit: (14073, 10) | Validation: (3601, 10)


In [81]:
def add_recent_hub_delay(target_df, history_df, window_hours=48):
    hist = history_df[["source_center", "od_end_time", "delay_ratio"]].copy()
    hist["end_ts"] = pd.to_datetime(hist["od_end_time"], format="mixed")
    hist = hist.sort_values("end_ts")
    groups = {hub: (g["end_ts"].to_numpy("datetime64[ns]"), g["delay_ratio"].to_numpy())
              for hub, g in hist.groupby("source_center")}
    window = np.timedelta64(window_hours, "h")
    starts = pd.to_datetime(target_df["od_start_time"]).to_numpy("datetime64[ns]")
    hubs = target_df["source_center"].to_numpy()

    med = np.full(len(target_df), np.nan)
    cnt = np.zeros(len(target_df), dtype=int)
    for i, (hub, s) in enumerate(zip(hubs, starts)):
        if hub not in groups:
            continue
        ends, delays = groups[hub]
        hi = np.searchsorted(ends, s, side="right")           # finished before departure
        lo = np.searchsorted(ends, s - window, side="left")   # within the last 48 hours
        if hi > lo:
            med[i] = np.median(delays[lo:hi])
            cnt[i] = hi - lo
    out = target_df.copy()
    out["recent_hub_median"] = med
    out["recent_hub_count"] = cnt
    return out

In [82]:
fit_r  = add_recent_hub_delay(fit_v2,  fit_df)
val_r  = add_recent_hub_delay(val_v2,  train)
cal_r  = add_recent_hub_delay(cal_v2,  pd.concat([train, cal]))
test_r = add_recent_hub_delay(test_v2, pd.concat([train, cal, test]))

print("Share of rows with recent hub history:",
      {n: round((d["recent_hub_count"] > 0).mean(), 3)
       for n, d in [("fit", fit_r), ("val", val_r), ("cal", cal_r), ("test", test_r)]})

Share of rows with recent hub history: {'fit': np.float64(0.84), 'val': np.float64(0.882), 'cal': np.float64(0.924), 'test': np.float64(0.9)}


In [83]:
from sklearn.metrics import mean_absolute_error

def run_residual(fit_frame, val_frame, seed=42):
    b_fit, b_val = base_pred(fit_frame), base_pred(val_frame)
    Xf, Xv = make_X(fit_frame), make_X(val_frame)
    m = XGBRegressor(
        objective="reg:absoluteerror", n_estimators=3000, learning_rate=0.03,
        max_depth=3, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=5, tree_method="hist", random_state=seed,
        eval_metric="mae", early_stopping_rounds=100)
    m.fit(Xf, y_fit.to_numpy() - b_fit,
          eval_set=[(Xv, y_val.to_numpy() - b_val)], verbose=False)
    pred = b_val + m.predict(Xv)
    return m, pred, b_val

m_plain,  pred_plain,  base_val = run_residual(fit_v2, val_v2)   # no recent features
m_recent, pred_recent, _        = run_residual(fit_r,  val_r)    # with recent hub delay

# sanity check: XGBoost's own score should now match yours
print("best_score (plain) :", round(m_plain.best_score, 4),
      "| recomputed:", round(mean_absolute_error(y_val, pred_plain), 4))
print("best_score (recent):", round(m_recent.best_score, 4),
      "| recomputed:", round(mean_absolute_error(y_val, pred_recent), 4))

ablation = pd.DataFrame({
    "Base alone (lane median)":            evaluate_model(y_val, base_val,    val_df["osrm_time_min"]),
    "Base + XGBoost (no recent)":          evaluate_model(y_val, pred_plain,  val_df["osrm_time_min"]),
    "Base + XGBoost (+ recent hub delay)": evaluate_model(y_val, pred_recent, val_df["osrm_time_min"]),
}).T
ablation

best_score (plain) : 0.6575 | recomputed: 0.6575
best_score (recent): 0.6575 | recomputed: 0.6575


,MAE_ratio,MedAE_ratio,MAE_excess_min
Base alone (lane median),0.6741,0.2500,33.19
Base + XGBoost (no recent),0.6575,0.2396,32.31
Base + XGBoost (+ recent hub delay),0.6575,0.2396,32.31


In [84]:
def paired_trip_bootstrap(y, p_new, p_base, trips, n=2000, seed=0):
    d = np.abs(y - p_base) - np.abs(y - p_new)      # positive = new is better
    g = pd.DataFrame({"trip": trips, "d": d}).groupby("trip")["d"].agg(["sum", "count"])
    s, c = g["sum"].to_numpy(), g["count"].to_numpy()
    rng = np.random.default_rng(seed)
    boots = [s[i].sum() / c[i].sum()
             for i in (rng.integers(0, len(g), len(g)) for _ in range(n))]
    return np.percentile(boots, [2.5, 50, 97.5])

y = y_val.to_numpy()
trips = val_df["trip_uuid"].to_numpy()
print("plain  vs base:", paired_trip_bootstrap(y, pred_plain,  base_val, trips))
print("recent vs base:", paired_trip_bootstrap(y, pred_recent, base_val, trips))

plain  vs base: [0.00972923 0.01655532 0.02342452]
recent vs base: [0.00972923 0.01655532 0.02342452]


In [85]:
val_start = val_df["od_start_time"].min()
late = pd.to_datetime(fit_df["od_end_time"], format="mixed") > val_start
print("Fit legs that finish after validation begins:", late.sum(), "of", len(fit_df))

Fit legs that finish after validation begins: 616 of 14073


In [86]:
def late_share(earlier, later, name):
    ends = pd.to_datetime(earlier["od_end_time"], format="mixed")
    boundary = pd.to_datetime(later["od_start_time"]).min()
    n = int((ends > boundary).sum())
    print(f"{name}: {n} of {len(earlier)} legs ({n/len(earlier):.1%}) finish after "
          f"the next pile's first departure ({boundary})")

late_share(fit_df, val_df, "fit -> val")
late_share(train, cal, "train -> cal")
late_share(pd.concat([train, cal]), test, "train+cal -> test")

fit -> val: 616 of 14073 legs (4.4%) finish after the next pile's first departure (2018-09-22 23:54:45.434611)
train -> cal: 781 of 17674 legs (4.4%) finish after the next pile's first departure (2018-09-26 00:00:11.712506)
train+cal -> test: 329 of 21975 legs (1.5%) finish after the next pile's first departure (2018-09-29 20:41:18.098697)


In [87]:
print("recent vs plain:", paired_trip_bootstrap(y, pred_recent, pred_plain, trips))

recent vs plain: [0. 0. 0.]


In [88]:
def make_X(d):
    X = build_features(d)[num_cols].copy()
    X["lane_base"] = base_pred(d)
    X["lane_history_count"] = d["lane_history_count"].to_numpy()
    for c in ["recent_hub_median", "recent_hub_count", "prev_delay_ratio", "prev_connected"]:
        if c in d.columns:
            X[c] = d[c].to_numpy()
    return X

def add_prev_leg_features(d):
    out = d.copy()
    out["_start"] = pd.to_datetime(out["od_start_time"])
    out["_end"] = pd.to_datetime(out["od_end_time"], format="mixed")
    out = out.sort_values(["trip_uuid", "_start"])
    g = out.groupby("trip_uuid")
    prev_end   = g["_end"].shift(1)
    prev_ratio = g["delay_ratio"].shift(1)
    prev_dest  = g["destination_center"].shift(1)
    ok = prev_end.notna() & (prev_end <= out["_start"]) # previous leg already finished
    out["prev_delay_ratio"] = prev_ratio.where(ok)
    out["prev_connected"] = ((prev_dest == out["source_center"]) & ok).astype(int)
    return out.drop(columns=["_start", "_end"]).loc[d.index]

fit_p, val_p = add_prev_leg_features(fit_r), add_prev_leg_features(val_r)
cal_p, test_p = add_prev_leg_features(cal_r), add_prev_leg_features(test_r)

print("Rows with a finished previous leg:", {n: round(d["prev_delay_ratio"].notna().mean(), 3) for n, d in [("fit", fit_p), ("val", val_p), ("cal", cal_p), ("test", test_p)]})

m_prev, pred_prev, _ = run_residual(fit_p, val_p)
print("best_score:", round(m_prev.best_score, 4), "| recomputed:", round(mean_absolute_error(y_val, pred_prev), 4))

ablation2 = pd.DataFrame({
    "Base alone": evaluate_model(y_val, base_val, val_df["osrm_time_min"]),
    "XGBoost (no recent)": evaluate_model(y_val, pred_plain, val_df["osrm_time_min"]),
    "XGBoost (+ recent hub)": evaluate_model(y_val, pred_recent, val_df["osrm_time_min"]),
    "XGBoost (+ recent + prev leg)": evaluate_model(y_val, pred_prev, val_df["osrm_time_min"]),
}).T
print(ablation2)
print("prev vs recent:", paired_trip_bootstrap(y, pred_prev, pred_recent, trips))

Rows with a finished previous leg: {'fit': np.float64(0.433), 'val': np.float64(0.446), 'cal': np.float64(0.436), 'test': np.float64(0.448)}
best_score: 0.659 | recomputed: 0.659
                               MAE_ratio  MedAE_ratio  MAE_excess_min
Base alone                        0.6741       0.2500           33.19
XGBoost (no recent)               0.6575       0.2396           32.31
XGBoost (+ recent hub)            0.6575       0.2396           32.31
XGBoost (+ recent + prev leg)     0.6590       0.2408           32.36
prev vs recent: [-0.0027574  -0.00147301 -0.00015701]


In [89]:
def final_eval(model, cal_frame, test_frame):
    results = {}

    for name, frame, actual in [
        ("calibration", cal_frame, cal["delay_ratio"]),
        ("test", test_frame, test["delay_ratio"])
    ]:
        base = base_pred(frame)
        prediction = base + model.predict(make_X(frame))

        results[f"{name} - lane baseline"] = evaluate_model(
            actual, base, frame["osrm_time_min"]
        )
        results[f"{name} - residual XGBoost"] = evaluate_model(
            actual, prediction, frame["osrm_time_min"]
        )

    return pd.DataFrame(results).T

final_model = m_plain
res = final_eval(final_model, cal_v2, test_v2)
display(res)

,MAE_ratio,MedAE_ratio,MAE_excess_min
calibration - lane baseline,0.6862,0.2500,30.03
calibration - residual XGBoost,0.6697,0.2426,30.01
test - lane baseline,0.6686,0.2615,30.42
test - residual XGBoost,0.6540,0.2611,30.04


In [90]:
import pandas as pd

enr = pd.read_csv("../data/processed/legs_enriched.csv")
legs = pd.read_csv("../data/processed/legs.csv")

print("Shape:", enr.shape, "| legs.csv shape:", legs.shape)
print("Same number of rows:", len(enr) == len(legs))

new_cols = [c for c in enr.columns if c not in legs.columns]
print("\nNew columns added by Person A:", new_cols)

keys = [c for c in ["trip_uuid", "source_center", "destination_center", "od_start_time"] if c in enr.columns]
print("Join keys available:", keys)
print("Duplicate keys:", enr.duplicated(subset=keys).sum())
print("\nShare missing per new column:")
print(enr[new_cols].isna().mean().round(3))
print("\nSummary of new columns:")
print(enr[new_cols].describe().T)

Shape: (7817, 27) | legs.csv shape: (26369, 14)
Same number of rows: False

New columns added by Person A: ['trip_created', 'split', 'departure_date', 'arrival_date', 'source_rain_mm', 'source_precipitation_mm', 'source_temperature_mean_c', 'source_weather_available', 'destination_rain_mm', 'destination_precipitation_mm', 'destination_temperature_mean_c', 'destination_weather_available', 'both_weather_available']
Join keys available: ['trip_uuid', 'source_center', 'destination_center', 'od_start_time']
Duplicate keys: 0

Share missing per new column:
trip_created                      0.000
split                             0.000
departure_date                    0.000
arrival_date                      0.000
source_rain_mm                    0.037
source_precipitation_mm           0.037
source_temperature_mean_c         0.037
source_weather_available          0.000
destination_rain_mm               0.048
destination_precipitation_mm      0.048
destination_temperature_mean_c    0.048
des

In [92]:
# def final_eval(model, cal_frame, test_frame):
#     out = {}
#     for name, frame, y_true in [("cal", cal_frame, cal["delay_ratio"]), ("test", test_frame, test["delay_ratio"])]:
#         base = base_pred(frame)
#         pred = base + model.predict(make_X(frame))
#         out[f"{name} : base alone"] = evaluate_model(y_true, base, frame["osrm_time_min"])
#         out[f"{name} : base + XGBoost"] = evaluate_model(y_true, pred, frame["osrm_time_min"])
#     return pd.DataFrame(out).T

# # Using m_recent configuration since it achieved the best MAE ratio (0.7125)
# print(final_eval(m_recent, cal_r, test_r))

# Skip legacy experimental evaluation.
# The final master-split evaluation is performed in Cell 76.

print("Skipping legacy evaluation. Using the master-split final model.")


Skipping legacy evaluation. Using the master-split final model.


In [93]:
# import json
# from pathlib import Path

# # Ensure directory exists
# Path("../experiments/day2").mkdir(parents=True, exist_ok=True)

# # Save evaluation results dictionary
# final_results_dict = {
#     "calibration_base_alone_mae_ratio": 0.6395,
#     "calibration_xgboost_mae_ratio": 0.6287,
#     "test_base_alone_mae_ratio": 0.7406,
#     "test_xgboost_mae_ratio": 0.7151
# }

# with open("../experiments/day2/final_eval_v1.json", "w") as f:
#     json.dump(final_results_dict, f, indent=4)

# # Save your winning residual XGBoost model
# m_recent.save_model("../experiments/day2/residual_xgb_v1.json")
# print("Successfully saved final evaluation JSON and XGBoost model!")

# Skip saving legacy experimental results.
# Updated master-split outputs are saved in Cells 81–83.

print("Skipping legacy experimental saves.")

Skipping legacy experimental saves.


In [94]:
enr = pd.read_csv("../data/processed/legs_enriched.csv")
mine = pd.concat([train.assign(my_split="train"), cal.assign(my_split="cal"), test.assign(my_split="test")])

k = ["trip_uuid", "source_center", "destination_center"]
m = enr.merge(mine[k + ["my_split"]], on=k, how="left")

print("Legs in weather file:", len(enr), "| hubs:",
      len(set(enr.source_center) | set(enr.destination_center)))
print("Matched to your piles:", m["my_split"].notna().sum(), "of", len(enr))
print(pd.crosstab(m["split"], m["my_split"]))

Legs in weather file: 7817 | hubs: 100
Matched to your piles: 7817 of 7817
my_split   cal  test  train
split                      
later     1237  1185      0
training     0     0   5395


In [95]:
import json
from pathlib import Path

EXP = Path("../experiments/day2/master_split")
MODELS = Path("../models")

EXP.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

model_path = MODELS / "residual_xgb_plain_master_split_v1.json"
results_path = EXP / "final_eval_master_split_v1.json"

assert not model_path.exists(), f"{model_path} already exists"
assert not results_path.exists(), f"{results_path} already exists"

final_model.save_model(str(model_path))
res.to_json(results_path, indent=2)

print("Model saved:", model_path)
print("Results saved:", results_path)

Model saved: ..\models\residual_xgb_plain_master_split_v1.json
Results saved: ..\experiments\day2\master_split\final_eval_master_split_v1.json


In [96]:
card = {
    "final_model": "Plain residual XGBoost",
    "description": "Lane median baseline + learned XGBoost correction",
    "model_file": "models/residual_xgb_plain_master_split_v1.json",
    "features": list(final_model.feature_names_in_),
    "best_iteration": int(final_model.best_iteration),
    "split_source": "data/processed/split_trip_ids.csv",
    "training": "Master training partition, internally divided chronologically into fit and validation",
    "calibration": "Master calibration partition",
    "test": "Master test partition",
    "results": res.round(4).to_dict(orient="index"),
    "notes": [
        "Recent-hub and previous-leg features are not included in the final model.",
        "Hub-ID categorical models are discarded.",
        "All learned model features and parameters are fitted using training data only."
    ]
}

card_path = EXP / "final_model_card_master_split_v1.json"
assert not card_path.exists(), f"{card_path} already exists"

with open(card_path, "w") as f:
    json.dump(card, f, indent=2)

print("Model card saved:", card_path)

Model card saved: ..\experiments\day2\master_split\final_model_card_master_split_v1.json


In [97]:
cols = [
    "trip_uuid",
    "source_center",
    "destination_center",
    "route_type",
    "od_start_time",
    "osrm_time_min",
    "osrm_distance_km",
    "lane_history_count",
    "delay_ratio"
]

def export_preds(model, frame, pile):
    out = frame[cols].copy()
    out["base_pred"] = base_pred(frame)
    out["pred_ratio"] = (
        out["base_pred"].to_numpy() +
        model.predict(make_X(frame))
    )
    out["pile"] = pile
    return out.rename(columns={"delay_ratio": "y_ratio"})

preds = pd.concat(
    [
        export_preds(final_model, cal_v2, "cal"),
        export_preds(final_model, test_v2, "test")
    ],
    ignore_index=True
)

for pile in ["cal", "test"]:
    d = preds[preds["pile"] == pile]
    print(
        pile,
        "| rows:", len(d),
        "| MAE:", round(
            (d["y_ratio"] - d["pred_ratio"]).abs().mean(), 4
        )
    )

pred_path = DATA / "preds_cal_test_master_split_v1.csv"
assert not pred_path.exists(), f"{pred_path} already exists"

preds.to_csv(pred_path, index=False)
print("Predictions saved:", pred_path)

cal | rows: 4301 | MAE: 0.6697
test | rows: 4394 | MAE: 0.654
Predictions saved: ..\data\processed\preds_cal_test_master_split_v1.csv


In [98]:
from pathlib import Path
import pandas as pd

pred_path = Path("../data/processed/preds_cal_test_master_split_v1.csv")

assert pred_path.exists(), "Prediction file not found!"

preds = pd.read_csv(pred_path)

print("Shape:", preds.shape)
print("\nColumns:")
print(preds.columns.tolist())

print("\nFirst 5 rows:")
display(preds.head())

print("\nMissing values:")
print(preds.isna().sum())

print("\nDuplicate rows:", preds.duplicated().sum())

Shape: (8695, 12)

Columns:
['trip_uuid', 'source_center', 'destination_center', 'route_type', 'od_start_time', 'osrm_time_min', 'osrm_distance_km', 'lane_history_count', 'y_ratio', 'base_pred', 'pred_ratio', 'pile']

First 5 rows:


,trip_uuid,source_center,destination_center,route_type,od_start_time,osrm_time_min,osrm_distance_km,lane_history_count,y_ratio,base_pred,pred_ratio,pile
0,trip-153792001171221733,IND500008AAF,IND500039AAC,Carting,2018-09-26 02:05:27.979102,17.0,20.8808,12,3.705882,3.345238,3.256235,cal
1,trip-153792001171221733,IND500039AAC,IND500035AAD,Carting,2018-09-26 00:00:11.712506,14.0,14.3507,4,3.142857,3.087413,2.745339,cal
2,trip-153792002908395866,IND160102AAC,IND160002AAC,Carting,2018-09-26 00:00:29.084196,35.0,44.7550,23,2.142857,2.000000,2.109860,cal
3,trip-153792008470813426,IND474003AAA,IND000000ACB,FTL,2018-09-26 00:01:24.708365,259.0,314.0100,11,1.474903,1.667939,1.694753,cal
4,trip-153792017810339021,IND400072AAB,IND401104AAA,Carting,2018-09-26 00:02:58.103647,15.0,19.3932,41,2.400000,2.941176,3.095553,cal



Missing values:
trip_uuid             0
source_center         0
destination_center    0
route_type            0
od_start_time         0
osrm_time_min         0
osrm_distance_km      0
lane_history_count    0
y_ratio               0
base_pred             0
pred_ratio            0
pile                  0
dtype: int64

Duplicate rows: 0


In [99]:
print("Partition counts:")
print(preds["pile"].value_counts())

print("\nUnique trips per partition:")
print(preds.groupby("pile")["trip_uuid"].nunique())

print("\nTrip overlap:")
cal_ids = set(preds.loc[preds["pile"] == "cal", "trip_uuid"])
test_ids = set(preds.loc[preds["pile"] == "test", "trip_uuid"])

print("Overlapping trips:", len(cal_ids & test_ids))

assert preds["pile"].value_counts().to_dict() == {
    "cal": 4301,
    "test": 4394
}
assert len(cal_ids & test_ids) == 0

print("\nAll partition checks passed!")

Partition counts:
pile
test    4394
cal     4301
Name: count, dtype: int64

Unique trips per partition:
pile
cal     2424
test    2424
Name: trip_uuid, dtype: int64

Trip overlap:
Overlapping trips: 0

All partition checks passed!
